In [0]:
-- Before we do the silver transformation, we need to verify the bronze layer data types:

DESCRIBE databricks_cnc_machine_project.cnc_machine_bronze_layer

-- In the next table we show the bronze layer data types (obtained using the results of the prevous query) and we decide what data type we want for the silver table:

/* Table with Data Types:

|Column                 |Bronze Data Type|Silver Data Type|
|-----------------------|----------------|----------------|
|UDI                    |bigint          |INT             |
|Product ID             |string          |STRING          |
|Type                   |string          |STRING          |
|Air temperature [K]    |double          |DOUBLE          |
|Process temperature [K]|double          |DOUBLE          |
|Rotational speed [rpm] |bigint          |INT             |
|Torque [Nm]            |double          |DOUBLE          |
|Tool wear [min]        |bigint          |INT             |
|Machine failure        |bigint          |BOOLEAN         |
|TWF                    |bigint          |BOOLEAN         |
|HDF                    |bigint          |BOOLEAN         |
|PWF                    |bigint          |BOOLEAN         |
|OSF                    |bigint          |BOOLEAN         |
|RNF                    |bigint          |BOOLEAN         |
|_load_timestamp        |timestamp       |N.A             |
*/

-- For columns like Product ID and Type, we maintain the string data type. Others, like UDI and Tool wear, we would like to change the data type from BIGINT to INT, optimizing storage. In the case of Machine failure and Failure Modes columns like TWF, HDF and so on, we want to change the data type from BIGINT to BOOLEAN.

-- We're also going to check if there are rows with ilogic or bad information in the bronze layer, running queries to identify them.

-- There are at least two scenarios: a machine failure is identified without a failure mode, or a failure mode is identified without a machine failure. This could happen for many reasons, for example, sensor failures or problems before or during the data ingestion process.


-- In this query we verify if there are rows with a machine failure without an identification for a failure mode:

SELECT *
FROM databricks_cnc_machine_project.cnc_machine_bronze_layer
WHERE `Machine failure` = 1 AND (`TWF` = 0 AND `HDF` = 0 AND `PWF` = 0 AND `OSF` = 0 AND `RNF` = 0)

-- For this dataset, there are 9 rows with this problem.

-- With the next query, we verify if there are rows that exhibit a failure mode without a machine failure:

SELECT *
FROM databricks_cnc_machine_project.cnc_machine_bronze_layer
WHERE `Machine failure` = 0 AND (`TWF` = 1 OR `HDF` = 1 OR `PWF` = 1 OR `OSF` = 1 OR `RNF` = 1)

-- There are 18 rows with the problem identified above. But, all 18 rows are related to RNF or Random Failure, which could lead us to think that could be a problem with the data upstream. For this reason, we run the next query to evaluate how many rows exhibit only random failures:

SELECT *
FROM databricks_cnc_machine_project.cnc_machine_bronze_layer
WHERE `RNF` = 1

-- There are in total 19 rows with random failure, almost indentical with those which doesn't exihibit machine failure. 
-- In this case, we decided to retain the data in the silver table for the 18 records of random failures that don't indicate a machine failure; this data may have been generated because a random failure doesn't necessarily mean that the machining process must end abruptly (for example, an alarm was triggered by a sensor, but the process was completed).

-- However, in the case of the 9 records that indicate a machine failure without specifying the failure mode, the decision has been made to remove them from the silver table and place them in a separate quarantine table, as it is highly likely that these are erroneous records or that they deserve further analysis to provide an alternative explanation (since additional failure modes may need to be identified).

-- Table created to quarantine records displaying a machine failure without a failure mode:

CREATE OR REPLACE TABLE databricks_cnc_machine_project.cnc_machine_silver_quarantine AS

(SELECT
  ROW_NUMBER() OVER (ORDER BY UDI) AS quarantine_id,
  UDI::INT AS original_udi,
  `Product ID` AS product_id,
  `Type` AS product_type,
  `Air temperature [K]` AS air_temperature_k,
  `Process temperature [K]` AS process_temperature_k,
  `Rotational speed [rpm]`::INT AS rotational_speed_rpm,
  `Torque [Nm]` AS torque_nm,
  `Tool wear [min]`::INT AS tool_wear_min,
  `Machine failure`::BOOLEAN AS machine_failure,
  TWF::BOOLEAN AS tool_wear_failure,
  HDF::BOOLEAN AS heat_dissipation_failure,
  PWF::BOOLEAN AS power_failure,
  OSF::BOOLEAN AS overstrain_failure,
  RNF::BOOLEAN AS random_failure,
  'Machine failure without a failure mode' AS quarantine_reason,
  current_timestamp() AS _quarantine_timestamp
FROM databricks_cnc_machine_project.cnc_machine_bronze_layer
WHERE `Machine failure` = 1 AND (`TWF` = 0 AND `HDF` = 0 AND `PWF` = 0 AND `OSF` = 0 AND `RNF` = 0))

-- The quarantine table has been created. Now, we can execute a general query on it:

SELECT *
FROM databricks_cnc_machine_project.cnc_machine_silver_quarantine




In [0]:
-- Next step, we run a transformation query for fixing column tags and data types. This helps with the workflow for analytics and visualization in the gold layer and dashboards.

-- Taking into account the data types from the bronze layer, we're replacing failure mode numbers 0's and 1's with boolean logic, optimizing storage and performance (0=no failure during the CNC operation; 1=machine failure and specific failure mode). We're also taking into account bigint to int data type transformation.

-- Lastly, we're calculating power consumption and adding a column with the timestamp of the transformation from bronze to silver layer for traceability.

------------------------------------------------------------------------------------

CREATE OR REPLACE TABLE databricks_cnc_machine_project.cnc_machine_silver_layer AS

(SELECT 
    row_number() OVER (ORDER BY UDI) AS silver_id,

    -- Changing column tags with optimized and legible description; transforming data types from bigint to int; applying decimal data type for better accuracy and precision in engineering calculations.
    UDI::INT AS original_udi,
    `Product ID` AS product_id,
    `Type` AS product_type,
    ROUND(`Air temperature [K]`, 2)::DECIMAL(10,2) AS air_temperature_k,
    ROUND(`Process temperature [K]`, 2)::DECIMAL(10,2) AS process_temperature_k,
    `Rotational speed [rpm]`::INT AS rotational_speed_rpm,
    ROUND(`Torque [Nm]`, 2)::DECIMAL(10,2) AS torque_nm,
    `Tool wear [min]`::INT AS tool_wear_min,

    -- Changing one or zero numbers by boolean logic:
    `Machine failure`::BOOLEAN AS machine_failure,
    TWF::BOOLEAN AS tool_wear_failure,
    HDF::BOOLEAN AS heat_dissipation_failure,
    PWF::BOOLEAN AS power_failure,
    OSF::BOOLEAN AS overstrain_failure,
    RNF::BOOLEAN AS random_failure,

    -- Power calculation (in Watts) for comparison and analytics in gold layer and dashboards (during the calculation rpm are transformed to rad/s):
    ROUND((`Torque [Nm]` * `Rotational speed [rpm]`) * (2 * pi() / 60), 2)::DECIMAL(10,2) AS power_watts,

    -- Silver layer transformation timestamp:
    current_timestamp() AS _silver_transformed_timestamp

FROM databricks_cnc_machine_project.cnc_machine_bronze_layer

-- In the next line, we are excluding records that exhibit a machine failure without a failure mode, as they were quarantined in a previous step:
WHERE NOT (`Machine failure` = 1 AND `TWF` = 0 AND `HDF` = 0 AND `PWF` = 0 AND `OSF` = 0 AND `RNF` = 0))

------------------------------------------------------------------------------------

-- After creating the silver table layer, we run a query to verify if the data was properly transformed:

SELECT *
FROM databricks_cnc_machine_project.cnc_machine_silver_layer


-- We can also verify the total number of records in the silver layer, which happens to be 9991 rows (10000 minus the 9 records excluded for the quarantine table):

SELECT count(*) AS total_silver 
FROM databricks_cnc_machine_project.cnc_machine_silver_layer


-- We verify, running the next query, that the data type for each column matches what was specified for the transformation from the Bronze to the Silver tier layer:

DESCRIBE databricks_cnc_machine_project.cnc_machine_silver_layer

/* Silver table data types:

|        col_name             |  data_type  |
|-----------------------------|-------------|
|silver_id                    |int          |
|original_udi                 |int          |
|product_id                   |string       |
|product_type                 |string       |
|air_temperature_k            |decimal(10,2)|
|process_temperature_k        |decimal(10,2)|
|rotational_speed_rpm         |int          |
|torque_nm                    |decimal(10,2)|
|tool_wear_min                |int          |
|machine_failure              |boolean      |
|tool_wear_failure            |boolean      |
|heat_dissipation_failure     |boolean      |
|power_failure                |boolean      |
|overstrain_failure           |boolean      |
|random_failure               |boolean      |
|power_watts                  |decimal(10,2)|
|_silver_transformed_timestamp|timestamp    |
*/

-- We can see that the conversion of machine failure and failure mode columns to Boolean values was successful, as was the conversion of variables such as temperature, torque, and power to decimal values. 


-----------------------------------------------------------
--  DESARROLLAR QUERIES PARA LAS TABLAS DE LA CAPA GOLD

---------------------------------------------------------------
-- Information where a machine failure occurs with only 1 failure mode:
SELECT
*,
-- We add up every failure mode (except random failures)
(tool_wear_failure::INT + 
heat_dissipation_failure::INT + 
power_failure::INT + 
overstrain_failure::INT) AS failure_modes_count

FROM databricks_cnc_machine_project.cnc_machine_silver_layer

-- Filter
WHERE machine_failure = True AND 
(tool_wear_failure::INT + 
heat_dissipation_failure::INT + 
power_failure::INT + 
overstrain_failure::INT) = 1


---------------------------------------------------------------
-- Information where a machine failure occurs with more than 1 failure mode:
SELECT
*,
-- We add up every failure mode (except random failures)
(tool_wear_failure::INT + 
heat_dissipation_failure::INT + 
power_failure::INT + 
overstrain_failure::INT) AS failure_modes_count

FROM databricks_cnc_machine_project.cnc_machine_silver_layer

-- Filter
WHERE machine_failure = True AND 
(tool_wear_failure::INT + 
heat_dissipation_failure::INT + 
power_failure::INT + 
overstrain_failure::INT) > 1


---------------------------------------------------------------

-- Summary of failed and succesful operations (since the values are Boolean, when you apply the simplified CAST function using ::, the value True returns a 1, which simplifies and optimizes the count):

SELECT

-- Total operations (with and without machine failure)
COUNT(*) AS total_operations,

-- Total succesful operations (Machine Failure = False)
SUM(CASE WHEN machine_failure = False THEN 1 ELSE 0 END) AS total_successful_operations,

-- Total failed operations (Machine Failure = True)
SUM(machine_failure::INT) AS total_machine_failures,

-- Total failure rate percentage
ROUND((SUM(machine_failure::INT) * 100.0) / COUNT(*), 2) AS total_failure_rate_percentage,

-- Total failures by failure mode:
SUM(`tool_wear_failure`::INT) AS total_tool_wear_failures,
ROUND((SUM(`tool_wear_failure`::INT) * 100.0) / COUNT(*), 2) AS percentage_tool_wear_failures,

SUM(`heat_dissipation_failure`::INT) AS total_heat_dissipation_failures,
ROUND((SUM(`heat_dissipation_failure`::INT) * 100.0) / COUNT(*), 2) AS percentage_heat_dissipation_failures,

SUM(`power_failure`::INT) AS total_power_failures,
ROUND((SUM(`power_failure`::INT) * 100.0) / COUNT(*), 2) AS percentage_power_failures,

SUM(`overstrain_failure`::INT) AS total_overstrain_failures,
ROUND((SUM(`overstrain_failure`::INT) * 100.0) / COUNT(*), 2) AS percentage_overstrain_failures,

SUM(`random_failure`::INT) AS total_random_failures,
ROUND((SUM(`random_failure`::INT) * 100.0) / COUNT(*), 2) AS percentage_random_failures

FROM databricks_cnc_machine_project.cnc_machine_silver_layer

-- Al correr esta query me da que existen un total de 330 fallas totales, pero al sumar cada uno de los modos de fallos (excepto random failure): 46(twf)+115(hdf)+95(pwf)+98(osf) = 354 fallas, es decir, que deben haber 24 fallas donde existe más de un modo de fallo.


------------------------------------------------------------------

-- Summary of failed and succesful operations by PRODUCT TYPE:

SELECT
product_type,

-- Total operations (with and without machine failure)
COUNT(*) AS total_operations,

-- Total succesful operations (Machine Failure = False)
SUM(CASE WHEN machine_failure = False THEN 1 ELSE 0 END) AS total_successful_operations,

-- Total failed operations (Machine Failure = True)
SUM(machine_failure::INT) AS total_machine_failures,

-- Failure percentage rate (%)
ROUND((SUM(machine_failure::INT) * 100.0) / COUNT(*), 2) AS failure_rate_percentage,

-- Tool Wear Failures
SUM(tool_wear_failure::INT) AS total_tool_wear_failures,
ROUND((SUM(tool_wear_failure::INT) * 100.0) / COUNT(*), 2) AS percent_tool_wear_failures,

-- Heat Dissipation Failures
SUM(heat_dissipation_failure::INT) AS total_heat_dissipation_failures,
ROUND((SUM(heat_dissipation_failure::INT) * 100.0) / COUNT(*), 2) AS percent_heat_dissipation_failures,

-- Power Failures
SUM(power_failure::INT) AS total_power_failures,
ROUND((SUM(power_failure::INT) * 100.0) / COUNT(*), 2) AS percent_power_failures,

-- Overstrain Failures
SUM(overstrain_failure::INT) AS total_overstrain_failures,
ROUND((SUM(overstrain_failure::INT) * 100.0) / COUNT(*), 2) AS percent_overstrain_failures,

-- Random Failures
SUM(random_failure::INT) AS total_random_failures,
ROUND((SUM(random_failure::INT) * 100.0) / COUNT(*), 2) AS percent_random_failures

FROM databricks_cnc_machine_project.cnc_machine_silver_layer
GROUP BY product_type
ORDER BY 
    CASE product_type
        WHEN 'L' THEN 1
        WHEN 'M' THEN 2
        WHEN 'H' THEN 3
        ELSE 4 
    END


---------------------------------------------------------------
-- Creo que deberia crear mejor una CTE que identifique los valores donde tool_wear_min=0, de tal forma usar en la consulta los datos de dicha CTE:

WITH table_toolwearmin_0 AS

(SELECT
*
FROM databricks_cnc_machine_project.cnc_machine_silver_layer
WHERE tool_wear_min = 0 AND silver_id > 1)

/*
-- Informacion una fila antes de cambiar la herramienta:
SELECT
*
FROM databricks_cnc_machine_project.cnc_machine_silver_layer
WHERE silver_id IN (SELECT (silver_id - 1) FROM table_toolwearmin_0)

-- Cantidad de veces que se cambia la herramienta:
SELECT
COUNT(*) AS tool_changes
FROM table_toolwearmin_0


-- Informacion una fila antes de cambiar la herramienta, donde se filtra para ver si la herramienta se cambia por una o multiples fallas, o sin falla:
SELECT *
FROM databricks_cnc_machine_project.cnc_machine_silver_layer
WHERE silver_id IN (SELECT (silver_id - 1) FROM table_toolwearmin_0) AND 
((machine_failure::INT +
tool_wear_failure::INT + 
heat_dissipation_failure::INT + 
power_failure::INT + 
overstrain_failure::INT) = 0) */

-- REALIZAR QUERY PARA OBTENER PROMEDIO DE TIEMPO DE CAMBIO DE HERRAMIENTA

SELECT
ROUND(AVG(tool_wear_min), 2) AS avg_tool_wear_min
FROM databricks_cnc_machine_project.cnc_machine_silver_layer
WHERE silver_id IN (SELECT (silver_id - 1) FROM table_toolwearmin_0)
